In [6]:

"""
compare_csvs.py — Compare two CSV files cell-by-cell and report how identical they are.

Works both as a plain script (python compare_csvs.py) and inside Jupyter

Setup: edit the SETTINGS block below.

Headers (the first row of each file) are never counted as data. Columns are
matched by header NAME, so the two files can have their columns in different
orders.
"""

import csv
import sys

# ============================================================
# SETTINGS:
#
# Columns to compare (must match the header names exactly):
#   COLUMNS = ["Conduct #1", "Result #1"]
#   COLUMNS = [] -- empty list means compare ALL columns
#
# Options (True or False):
#   SHOW_DIFFS  — list every differing cell
#   IGNORE_CASE — treat 'Yes' and 'yes' as identical
#   STRIP       — ignore leading/trailing spaces in cells
# ============================================================
FILE1 = '/Users/chris/Desktop/To test CSV agreement/Chris/DUI/structured_mensrea_output_model=gpt-5.6-sol_DUI.csv'
FILE2 =  '/Users/chris/Desktop/To test CSV agreement/Daniel/DUI/DUI=gpt-5.6-sol_DUI.csv'
COLUMNS = ["Conduct #1", "Conduct #2", "Conduct #3",
           "Result #1", "Result #2", "Result #3",
           "Circumstance #1", "Circumstance #2", "Circumstance #3"]

SHOW_DIFFS = True
IGNORE_CASE = True
STRIP = True


def read_csv(path):
    """Read a CSV file. Returns (header, data_rows)."""
    try:
        with open(path, newline="", encoding="utf-8-sig") as f:
            rows = list(csv.reader(f))
    except FileNotFoundError:
        sys.exit(f"Error: file not found: {path}")
    except Exception as e:
        sys.exit(f"Error reading {path}: {e}")

    if not rows:
        sys.exit(f"Error: {path} is empty.")

    return rows[0], rows[1:]


def column_indices(header, wanted, path):
    """Map each wanted column name to its index in this file's header.

    If wanted is empty, use every column in the header.
    """
    if not wanted:
        return list(header), {name: i for i, name in enumerate(header)}

    indices = {}
    missing = []
    for name in wanted:
        if name in header:
            indices[name] = header.index(name)
        else:
            missing.append(name)

    if missing:
        sys.exit(
            f"Error: column(s) not found in {path}: {missing}\n"
            f"Available columns: {header}"
        )

    return list(wanted), indices


def get_cell(row, idx):
    """Safely get a cell from a row (returns '' if the row is short)."""
    return row[idx] if idx < len(row) else ""


def compare(rows1, rows2, cols, idx1, idx2, ignore_case=False, strip=False):
    """Compare the selected columns cell-by-cell over the overlapping rows.

    Returns (total_cells, identical, differences) where differences is a list
    of (data_row_number, column_name, value1, value2), 1-based row numbers.
    """
    n_rows = min(len(rows1), len(rows2))
    differences = []
    total = 0
    identical = 0

    for i in range(n_rows):
        for name in cols:
            v1 = get_cell(rows1[i], idx1[name])
            v2 = get_cell(rows2[i], idx2[name])
            a, b = v1, v2
            if strip:
                a, b = a.strip(), b.strip()
            if ignore_case:
                a, b = a.lower(), b.lower()
            total += 1
            if a == b:
                identical += 1
            else:
                differences.append((i + 1, name, v1, v2))

    return total, identical, differences


def main():
    print(f"Comparing:\n  {FILE1}\n  {FILE2}\n")
    header1, rows1 = read_csv(FILE1)
    header2, rows2 = read_csv(FILE2)

    cols, idx1 = column_indices(header1, COLUMNS, FILE1)
    _, idx2 = column_indices(header2, cols, FILE2)

    if COLUMNS:
        print(f"Columns under analysis: {cols}\n")

    if len(rows1) != len(rows2):
        print(
            f"Warning: files have different numbers of data rows "
            f"({len(rows1)} vs {len(rows2)}). Comparing the first "
            f"{min(len(rows1), len(rows2))} rows only.\n"
        )

    total, identical, differences = compare(
        rows1, rows2, cols, idx1, idx2,
        ignore_case=IGNORE_CASE, strip=STRIP
    )

    if total == 0:
        sys.exit("No overlapping data cells to compare.")

    different = total - identical
    pct_identical = 100 * identical / total
    pct_different = 100 * different / total

    print(f"Cells compared:   {total}")
    print(f"Identical cells:  {identical}")
    print(f"Different cells:  {different}")
    print(f"% identical:      {pct_identical:.2f}%")
    print(f"% different:      {pct_different:.2f}%")

    if SHOW_DIFFS and differences:
        print("\nDiffering cells (data row, column):")
        for row, col_name, v1, v2 in differences:
            print(f"  Row {row}, Column '{col_name}': {v1!r} vs {v2!r}")


if __name__ == "__main__":
    main()

Comparing:
  /Users/chris/Desktop/To test CSV agreement/Chris/DUI/structured_mensrea_output_model=gpt-5.6-sol_DUI.csv
  /Users/chris/Desktop/To test CSV agreement/Daniel/DUI/DUI=gpt-5.6-sol_DUI.csv

Columns under analysis: ['Conduct #1', 'Conduct #2', 'Conduct #3', 'Result #1', 'Result #2', 'Result #3', 'Circumstance #1', 'Circumstance #2', 'Circumstance #3']

Cells compared:   459
Identical cells:  459
Different cells:  0
% identical:      100.00%
% different:      0.00%
